# 04. Train a small decision model for free

We now fine-tune a small open base model on the JSONL produced by notebook 03. The training code uses LoRA, which updates a small adapter instead of the whole model, so it runs on modest hardware.

Install the optional extra first with `uv sync --extra train`. Only torch, transformers, peft, and accelerate are added. No paid service is involved.

The approach follows free open work such as Tev1, minimind-jev, and Strands Decider.


<div align='center'><img src='../images/training_pipeline.png' alt='Training pipeline' style='width:80%; max-width:850px;'></div>

*The JSONL from notebook 03 trains a small LoRA adapter on top of an open base model. The adapter is saved locally and stays small.*


## Step 1. Check whether the training extra is available

We try to import torch. If it is missing, the cell prints the exact command to install it instead of raising an error. This keeps the notebook readable before installation.


In [ ]:
try:
    import torch
    print('torch', torch.__version__)
except ImportError:
    print('run: uv sync --extra train')


## Step 2. Load the converted records

Notebook 03 must have run first. We read the JSONL and count the records so we know how many training examples we have.


In [ ]:
from decision_qa import read_jsonl
records = read_jsonl('data/pubmedqa_decisions.jsonl')
print(len(records))


In [ ]:
assert len(records) > 0
print('dataset loaded')


## Step 3. Build text pairs

The training helper selects the Choice question from each record and turns it into a prompt plus a one-token answer. We inspect the result before training.


In [ ]:
from decision_qa.train import build_text_pairs
pairs = build_text_pairs(records)
print(len(pairs))


In [ ]:
prompt, answer = pairs[0]
print(prompt[:400])


In [ ]:
print(repr(answer))
assert answer.strip() in 'ABCDEFGHIJ'


## Step 4. Why we mask the prompt

During training we compute loss only on the answer token. The prompt tokens are labelled with -100 so the model is not rewarded for memorising the input. This is standard causal language model fine-tuning and keeps the example small.


## Step 5. Training settings

The base model is open and small. LoRA adds a rank-16 adapter on the attention projections. One epoch is enough for a demonstration; a real thesis experiment would sweep learning rate and epochs on a held-out split.


In [ ]:
import os
base_model = os.getenv('DM_BASE_MODEL', 'Qwen/Qwen2.5-0.5B')
print(base_model)


In [ ]:
assert 'Qwen' in base_model
print('open base model selected')


## Step 6. Run the training, when ready

The flag is off by default so the notebook is safe to run on any machine. Turn it on after installing the extra. The adapter is saved locally under `checkpoints/`, which is gitignored.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from decision_qa.train import train
    adapter = train('data/pubmedqa_decisions.jsonl', 'checkpoints/dm-lora', epochs=1)
    print('saved', adapter)
else:
    print('training skipped on purpose')


## Step 7. What a full decision model changes

This LoRA version writes a single answer token. A full decision model, as in Tev1 or Strands Decider, replaces the language head with a scorer over the declared options. Both produce a distribution; the scorer does it in one forward pass without decoding.

After training, rerun notebook 05 to compare the adapter with the local open model and with the free hosted route. Keep all resources free.
